# Comparaison TF-IDF vs BM25 vs Jaccard — Top 10

Ce notebook utilise `product.csv` et `query.csv`. Il prend les 3 premières requêtes et compare les 3 méthodes.

In [ ]:
import math
import re
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd


## 1) Charger les fichiers

In [ ]:
def find_file(filename):
    candidates = [
        Path(filename),
        Path("/mnt/data") / filename,
    ]

    if filename == "product.csv":
        candidates += [Path("product(1).csv"), Path("/mnt/data") / "product(1).csv"]

    if filename == "query.csv":
        candidates += [Path("query(1).csv"), Path("/mnt/data") / "query(1).csv"]

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(f"Fichier introuvable : {filename}")


product_df = pd.read_csv(find_file("product.csv"), sep="\t")
query_df = pd.read_csv(find_file("query.csv"), sep="\t")

query_df_3 = query_df.head(3).copy()

print("Nombre de produits :", len(product_df))
print("Nombre total de queries :", len(query_df))
display(query_df_3[["query_id", "query", "query_class"]])


## 2) Préparer le texte des produits

In [ ]:
text_columns = [
    "product_name",
    "product_class",
    "category hierarchy",
    "product_description",
]

for col in text_columns:
    product_df[col] = product_df[col].fillna("").astype(str)

product_df["document_text"] = product_df[text_columns].agg(" ".join, axis=1)

display(product_df[["product_id", "product_name", "product_class", "document_text"]].head())


## 3) Nettoyer le texte

In [ ]:
STOP_WORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from",
    "in", "is", "it", "of", "on", "or", "that", "the", "this", "to",
    "with", "you", "your", "can", "will", "its", "have", "has", "had",
    "but", "not", "into", "more", "than", "such"
}


def preprocess(text):
    if pd.isna(text):
        return []

    text = str(text).lower()
    tokens = re.findall(r"[a-z0-9]+", text)

    return [
        token
        for token in tokens
        if len(token) > 1 and token not in STOP_WORDS
    ]


print(preprocess("Smart Coffee Table!"))


## 4) Construire l'index inversé

In [ ]:
inverted_index = defaultdict(dict)
document_lengths = {}

for _, row in product_df.iterrows():
    product_id = row["product_id"]
    tokens = preprocess(row["document_text"])

    word_counts = Counter(tokens)
    document_lengths[product_id] = len(tokens)

    for word, frequency in word_counts.items():
        inverted_index[word][product_id] = frequency

print("Nombre de mots différents :", len(inverted_index))
print("Exemple pour 'chair' :", list(inverted_index.get("chair", {}).items())[:10])


## 5) Afficher les mots les plus fréquents

In [ ]:
def get_word_frequency_table(inverted_index, top_n=30):
    rows = []

    for word, postings in inverted_index.items():
        rows.append(
            {
                "word": word,
                "total_frequency": sum(postings.values()),
                "document_frequency": len(postings),
            }
        )

    frequency_df = pd.DataFrame(rows)

    if frequency_df.empty:
        return frequency_df

    return frequency_df.sort_values("total_frequency", ascending=False).head(top_n)


word_frequency_df = get_word_frequency_table(inverted_index, top_n=30)
display(word_frequency_df)


## 6) Préparer IDF pour TF-IDF et BM25

In [ ]:
total_documents = len(product_df)

idf_tfidf = {}
idf_bm25 = {}

for word, postings in inverted_index.items():
    df = len(postings)

    idf_tfidf[word] = math.log((total_documents + 1) / (df + 1)) + 1
    idf_bm25[word] = math.log(1 + (total_documents - df + 0.5) / (df + 0.5))

average_document_length = sum(document_lengths.values()) / len(document_lengths)

print("IDF TF-IDF chair :", idf_tfidf.get("chair"))
print("IDF BM25 chair :", idf_bm25.get("chair"))
print("Longueur moyenne document :", average_document_length)


## 7) Recherche TF-IDF

In [ ]:
def search_tfidf(query, top_n=10):
    query_terms = preprocess(query)
    scores = defaultdict(float)

    for term in query_terms:
        if term not in inverted_index:
            continue

        postings = inverted_index[term]

        for product_id, frequency in postings.items():
            document_length = document_lengths[product_id] or 1

            tf = frequency / document_length
            score = tf * idf_tfidf[term]

            scores[product_id] += score

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]


## 8) Recherche BM25

In [ ]:
def search_bm25(query, top_n=10, k1=1.5, b=0.75):
    query_counts = Counter(preprocess(query))
    scores = defaultdict(float)

    for term, query_frequency in query_counts.items():
        if term not in inverted_index:
            continue

        postings = inverted_index[term]

        for product_id, frequency in postings.items():
            document_length = document_lengths[product_id] or 1

            denominator = frequency + k1 * (
                1 - b + b * (document_length / average_document_length)
            )

            bm25_score = idf_bm25[term] * (
                (frequency * (k1 + 1)) / denominator
            )

            scores[product_id] += query_frequency * bm25_score

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]


## 9) Recherche Jaccard

In [ ]:
def jaccard_similarity(query, document_text):
    query_words = set(preprocess(query))
    document_words = set(preprocess(document_text))

    if not query_words or not document_words:
        return 0.0

    intersection = query_words.intersection(document_words)
    union = query_words.union(document_words)

    return len(intersection) / len(union)


def search_jaccard(query, top_n=10):
    scores = []

    for _, row in product_df.iterrows():
        product_id = row["product_id"]
        document_text = row["document_text"]

        score = jaccard_similarity(query, document_text)

        scores.append((product_id, score))

    return sorted(scores, key=lambda x: x[1], reverse=True)[:top_n]


## 10) Créer le tableau de comparaison

In [ ]:
product_by_id = product_df.set_index("product_id", drop=False)


def get_query_terms_frequency(query, product_id):
    terms = preprocess(query)

    parts = []
    for term in terms:
        frequency = inverted_index.get(term, {}).get(product_id, 0)
        parts.append(f"{term}:{frequency}")

    return " | ".join(parts)


def build_comparison_table(query_df_3, top_n=10):
    rows = []

    for _, query_row in query_df_3.iterrows():
        query_id = query_row["query_id"]
        query = query_row["query"]

        methods = {
            "TF-IDF": search_tfidf(query, top_n=top_n),
            "BM25": search_bm25(query, top_n=top_n),
            "Jaccard": search_jaccard(query, top_n=top_n),
        }

        for method_name, results in methods.items():
            for rank, (product_id, score) in enumerate(results, start=1):
                product = product_by_id.loc[product_id]

                rows.append(
                    {
                        "query_id": query_id,
                        "query": query,
                        "method": method_name,
                        "rank": rank,
                        "product_id": product_id,
                        "score": round(float(score), 6),
                        "product_name": product["product_name"],
                        "product_class": product["product_class"],
                        "query_term_frequencies": get_query_terms_frequency(query, product_id),
                        "preview": str(product["product_description"])[:160],
                    }
                )

    return pd.DataFrame(rows)


## 11) Lancer la comparaison Top 10

In [ ]:
comparison_df = build_comparison_table(query_df_3, top_n=10)

display(comparison_df)


## 12) Voir séparément chaque query

In [ ]:
for query in query_df_3["query"]:
    print("=" * 100)
    print("QUERY :", query)

    display(
        comparison_df[comparison_df["query"] == query]
        .sort_values(["method", "rank"])
    )


## 13) Sauvegarder les résultats

In [ ]:
comparison_df.to_csv("resultats_tfidf_bm25_jaccard_top10.csv", index=False)

print("Fichier sauvegardé : resultats_tfidf_bm25_jaccard_top10.csv")
